In [0]:
%run "./00_parametros_e_variaveis"

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS anvisa
COMMENT 'Schema unificado para o projeto Anvisa (Hands-on Mackenzie)';

In [0]:
from pyspark.sql.types import StructType, StructField, StringType
from pyspark.sql.functions import current_timestamp, col

# Garante a existência do schema no catálogo atual
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {schema_nome}")

# Relação de tabelas com prefixo b_
arquivos_para_ingerir = {
    "DADOS_ABERTOS_MEDICAMENTOS.csv": "b_dados_abertos_medicamentos",
    "VigiMed_Medicamentos.csv": "b_vigimed_medicamentos",
    "VigiMed_Notificacoes.csv": "b_vigimed_notificacoes",
    "VigiMed_Reacoes.csv": "b_vigimed_reacoes"
}

# Pipeline de Ingestão
for arquivo_csv, tabela_nome in arquivos_para_ingerir.items():
    caminho_origem_csv = f"{workspace_user_path}/{arquivo_csv}"
    caminho_destino_bronze = (
        f"abfss://{container}@{storage_account}.dfs.core.windows.net/"
        f"{projeto_dir}/01_bronze/{tabela_nome}"
    )
    
    print(f"--- Processando: {arquivo_csv} -> {schema_nome}.{tabela_nome} ---")
    
    # Passo A: Extrai a lista de colunas a partir do cabeçalho
    df_temp = (
        spark.read.format("csv")
        .option("header", "true")
        .option("sep", ";")
        .load(caminho_origem_csv)
    )
    
    # Passo B: Monta schema forçando 100% StringType
    schema_all_string = StructType([
        StructField(col_name, StringType(), True) 
        for col_name in df_temp.columns
    ])
    
    # Passo C: Leitura completa aplicando o schema explícito
    df_raw = (
        spark.read.format("csv")
        .option("header", "true")
        .option("sep", ";")
        .option("encoding", "ISO-8859-1")
        .schema(schema_all_string)
        .load(caminho_origem_csv)
    )
    
    # Passo D: Metadados técnicos suportados no Unity Catalog
    df_bronze = (
        df_raw
        .withColumn("_data_ingestao", current_timestamp())
        .withColumn("_arquivo_origem", col("_metadata.file_path"))
    )
    
    # Passo E: Grava fisicamente no ADLS Gen2 na pasta 01_bronze
    (
        df_bronze.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .save(caminho_destino_bronze)
    )
    
    # Passo F: Registra como tabela no catálogo Unity sem conflito de External Location
    (
        df_bronze.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(f"{schema_nome}.{tabela_nome}")
    )
    
    print(f"Sucesso: Tabela '{schema_nome}.{tabela_nome}' registrada e arquivos salvos em '{caminho_destino_bronze}'\n")

In [0]:
%sql
-- SELECT * FROM anvisa.b_dados_abertos_medicamentos LIMIT 10;
-- SELECT * FROM anvisa.b_vigimed_medicamentos LIMIT 10;
-- SELECT * FROM anvisa.b_vigimed_reacoes LIMIT 10;
-- SELECT * FROM anvisa.b_vigimed_notificacoes LIMIT 10;